In [9]:
   DATA = "bookings.jsonl"

In [10]:
import json
from datetime import date, datetime
from typing import Literal


import pandas as pd
from pydantic import BaseModel, Field, ValidationError, model_validator

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)

In [11]:
raw_bookings = []     # (line_number, parsed dict) for lines that are valid JSON
quarantine = []       # every rejected line goes here, with a reason
lines_read = 0

with open(DATA, encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        if not line.strip():              # skip blank lines
            continue
        lines_read += 1
        try:
            doc = json.loads(line)
        except json.JSONDecodeError:
            quarantine.append({"booking_id": None, "line": line_no,
                               "reason": "invalid JSON"})
            continue
        if not isinstance(doc, dict):     # valid JSON but not an object
            quarantine.append({"booking_id": None, "line": line_no,
                               "reason": "not a JSON object"})
            continue
        raw_bookings.append((line_no, doc))

print(f"{lines_read} lines read")
print(f"{len(raw_bookings)} parsed, {len(quarantine)} quarantined so far")

283 lines read
283 parsed, 0 quarantined so far


In [12]:
class Student(BaseModel):
    id: str = Field(pattern=r"^\d{2}-\d{4}$")
    name: str = Field(min_length=1)
    faculty: Literal["MET", "IET", "Management", "EMS","Pharmacy"]
   


In [13]:

class Ticket(BaseModel):
    type: Literal["standard", "vip", "student"]
    qty : int = Field(ge=1)          # ge: greater than or equal to
    price: float = Field(ge=0)
    

In [14]:

class Event(BaseModel):
    id: str = Field(pattern=r"^EV-\d{2}$")
    title: str = Field(min_length=1)
    club: Literal["Robotics","Debate","Music","Football","Chess"]
    date: date
    venue: str = Field(min_length=1)

In [ ]:
class Booking(BaseModel):
    booking_id: str = Field(pattern=r"^B\d{4}$")
    booked_at: datetime     
    student:Student
    event:Event
    tickets: list[Ticket] = Field(min_length=1)
    status: Literal["confirmed", "cancelled"]   
    total: float
    payment: Literal["cash", "card", "wallet"]
    
   
    @model_validator(mode="after")
    def check_total(self):
        expected = sum(t.qty * t.price for t in self.tickets)
        if abs(self.total - expected) > 0.01:
            raise ValueError("total does not match sum of ticket prices")
        return self
       
    @model_validator(mode="after")
    def check_booking_date(self):
        if self.booked_at.date() >= self.event.date:
            raise ValueError("booked on or after the event date")
        return self

In [16]:
def make_reason(err):
    """One Pydantic error -> a short, stable text (no bad values in it)."""
    where = ".".join(str(p) for p in err["loc"])
    msg = err["msg"].removeprefix("Value error, ")
    return f"{where}: {msg}" if where else msg

valid = []   # (line_no, Booking)

for line_no, doc in raw_bookings:
    try:
        valid.append((line_no, Booking.model_validate(doc)))
    except ValidationError as e:
        errors = e.errors()
        quarantine.append({
            "booking_id": doc.get("booking_id"),
            "line": line_no,
            "reason": make_reason(errors[0]),               # main reason, used for counting
            "details": [make_reason(x) for x in errors],    # every problem
            "record": doc,                                  # the raw line, so it can be replayed
        })

print(f"{len(valid)} passed validation, {len(quarantine)} quarantined so far")

AttributeError: 'str' object has no attribute 'date'

In [ ]:
seen = set()
bookings_ok = []   # valid, unique Booking objects

for line_no, booking in valid:
    if booking.booking_id in seen:
        quarantine.append({
            "booking_id": booking.booking_id,
            "line": line_no,
            "reason": "duplicate booking_id",
            "details": ["duplicate booking_id"],
            "record": booking.model_dump(mode="json"),
        })
    else:
        seen.add(booking.booking_id)
        bookings_ok.append(booking)

print(f"{len(bookings_ok)} bookings left, {len(quarantine)} quarantined in total")

In [ ]:
with open("quarantine.jsonl", "w", encoding="utf-8") as f:
    for q in quarantine:
        f.write(json.dumps(q, ensure_ascii=False) + "\n")

In [ ]:
from collections import Counter

reason_counts = Counter(q["reason"] for q in quarantine)

print(f"Lines read:   {lines_read}")
print(f"Passed:       {len(bookings_ok)}")
print(f"Quarantined:  {len(quarantine)}")
print("\nQuarantined by reason:")
for reason, n in reason_counts.most_common():
    print(f"  {n:>4}  {reason}")

# nothing lost: every line is either kept or quarantined
assert len(bookings_ok) + len(quarantine) == lines_read